In [3]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path.cwd().parent

SALES_DIR = (
    BASE_DIR
    / "data"
    / "synthetic"
)

sales_files = sorted(
    SALES_DIR.glob("sales_*.csv")
)

total_rows = 0
valid_file_count = 0
all_columns = set()

status_values = set()

for file in sales_files:
    df = pd.read_csv(file)

    total_rows += len(df)
    valid_file_count += 1

    all_columns.update(df.columns)

    if "order_status" in df.columns:
        status_values.update(
            df["order_status"].dropna().astype(str).unique()
        )

required_columns = [
    "sales_line_id",
    "order_id",
    "order_date",
    "quantity",
    "unit_selling_price",
    "discount_pct",
    "order_status"
]

missing_columns = [
    col for col in required_columns
    if col not in all_columns
]

order_status_exists = "order_status" in all_columns

print("02.16 STEP 1 — CANCELLATION DATA INSPECTION")
print("=" * 70)

print("Sales files found:", len(sales_files))
print("Total sales rows:", total_rows)

print("\nRequired Column Check")
print("-" * 70)

print("Missing required columns:", missing_columns)
print("Order status column exists:", order_status_exists)

print("\nOrder Status Values")
print("-" * 70)

if order_status_exists:
    print(sorted(status_values))
else:
    print("order_status column not found")

print("\nValidation")
print("-" * 70)

files_valid = len(sales_files) == 32
rows_valid = total_rows == 2_118_200
columns_valid = len(missing_columns) == 0
status_valid = order_status_exists

print("32 sales files:", files_valid)
print("2,118,200 sales rows:", rows_valid)
print("Required columns present:", columns_valid)
print("Order status exists:", status_valid)

if (
    files_valid
    and rows_valid
    and columns_valid
    and status_valid
):
    print("\n02.16 Step 1 inspection: PASSED")
else:
    print("\n02.16 Step 1 inspection: FAILED")

02.16 STEP 1 — CANCELLATION DATA INSPECTION
Sales files found: 32
Total sales rows: 2118200

Required Column Check
----------------------------------------------------------------------
Missing required columns: []
Order status column exists: True

Order Status Values
----------------------------------------------------------------------
['Cancelled', 'Completed']

Validation
----------------------------------------------------------------------
32 sales files: True
2,118,200 sales rows: True
Required columns present: True
Order status exists: True

02.16 Step 1 inspection: PASSED


In [4]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path.cwd().parent

SALES_DIR = (
    BASE_DIR
    / "data"
    / "synthetic"
)

sales_files = sorted(
    SALES_DIR.glob("sales_*.csv")
)

sales_list = []

for file in sales_files:
    df = pd.read_csv(file)

    sales_list.append(
        df[
            [
                "sales_line_id",
                "order_id",
                "order_date",
                "quantity",
                "unit_selling_price",
                "discount_pct",
                "order_status"
            ]
        ]
    )

sales = pd.concat(
    sales_list,
    ignore_index=True
)

print("02.16 STEP 2 — CURRENT CANCELLATION BEHAVIOR")
print("=" * 70)

status_summary = (
    sales["order_status"]
    .value_counts()
    .rename_axis("order_status")
    .reset_index(name="sales_lines")
)

status_summary["share_pct"] = (
    status_summary["sales_lines"]
    / len(sales)
    * 100
)

print("\nSales-line status distribution:")
print(status_summary.to_string(index=False))

print("\nUnique orders by status:")

order_status_summary = (
    sales.groupby("order_status")["order_id"]
    .nunique()
    .reset_index(name="unique_orders")
)

order_status_summary["share_pct"] = (
    order_status_summary["unique_orders"]
    / sales["order_id"].nunique()
    * 100
)

print(order_status_summary.to_string(index=False))

print("\nTotal unique orders:", sales["order_id"].nunique())

print("\nCancellation check:")
cancelled_lines = (
    sales["order_status"]
    .eq("Cancelled")
    .sum()
)

cancelled_orders = (
    sales.loc[
        sales["order_status"].eq("Cancelled"),
        "order_id"
    ]
    .nunique()
)

print("Cancelled sales lines:", cancelled_lines)
print("Cancelled unique orders:", cancelled_orders)

02.16 STEP 2 — CURRENT CANCELLATION BEHAVIOR

Sales-line status distribution:
order_status  sales_lines  share_pct
   Completed      2033443  95.998631
   Cancelled        84757   4.001369

Unique orders by status:
order_status  unique_orders  share_pct
   Cancelled          83944   4.915980
   Completed        1652542  96.777182

Total unique orders: 1707574

Cancellation check:
Cancelled sales lines: 84757
Cancelled unique orders: 83944


In [6]:
from pathlib import Path
import pandas as pd
import numpy as np

BASE_DIR = Path.cwd().parent

INPUT_DIR = (
    BASE_DIR
    / "data"
    / "synthetic"
    / "pricing_discount_adjusted"
    / "sales_adjusted"
)

OUTPUT_DIR = (
    BASE_DIR
    / "data"
    / "synthetic"
    / "cancellation_adjusted"
    / "sales_adjusted"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sales_files = sorted(INPUT_DIR.glob("sales_*.csv"))

print("02.16 STEP 3 — CORRECT CANCELLATION BEHAVIOR")
print("=" * 70)

# ---------------------------------------------------------
# Load all sales files
# ---------------------------------------------------------

sales_list = []

for file in sales_files:
    df = pd.read_csv(file)
    sales_list.append(df)

sales = pd.concat(
    sales_list,
    ignore_index=True
)

print("Sales files:", len(sales_files))
print("Total sales rows:", len(sales))

# ---------------------------------------------------------
# Target cancellation behavior
# ---------------------------------------------------------

TOTAL_CANCELLATIONS = int(
    round(len(sales) * 0.04)
)

ONLINE_TARGET_RATE = 0.06

online_mask = sales["sales_channel"].eq("Online")
store_mask = sales["sales_channel"].eq("Store")

online_count = online_mask.sum()
store_count = store_mask.sum()

# Number of online cancellations
online_cancel_count = int(
    round(online_count * ONLINE_TARGET_RATE)
)

# Remaining cancellations go to Store
store_cancel_count = (
    TOTAL_CANCELLATIONS
    - online_cancel_count
)

# ---------------------------------------------------------
# Reset statuses
# ---------------------------------------------------------

sales["order_status"] = "Completed"

# ---------------------------------------------------------
# Deterministic random selection
# ---------------------------------------------------------

rng = np.random.default_rng(42)

online_indices = sales.index[online_mask].to_numpy()
store_indices = sales.index[store_mask].to_numpy()

online_cancel_indices = rng.choice(
    online_indices,
    size=online_cancel_count,
    replace=False
)

store_cancel_indices = rng.choice(
    store_indices,
    size=store_cancel_count,
    replace=False
)

sales.loc[
    online_cancel_indices,
    "order_status"
] = "Cancelled"

sales.loc[
    store_cancel_indices,
    "order_status"
] = "Cancelled"

# ---------------------------------------------------------
# Save files back month by month
# ---------------------------------------------------------

for file in sales_files:

    df = pd.read_csv(file)

    file_ids = set(df["sales_line_id"])

    output_df = sales[
        sales["sales_line_id"].isin(file_ids)
    ].copy()

    output_df = output_df[
        df.columns
    ]

    output_file = OUTPUT_DIR / file.name

    output_df.to_csv(
        output_file,
        index=False
    )

# ---------------------------------------------------------
# Final validation
# ---------------------------------------------------------

cancelled = sales[
    sales["order_status"] == "Cancelled"
]

completed = sales[
    sales["order_status"] == "Completed"
]

online_total = online_count
store_total = store_count

online_cancelled = (
    cancelled["sales_channel"]
    .eq("Online")
    .sum()
)

store_cancelled = (
    cancelled["sales_channel"]
    .eq("Store")
    .sum()
)

online_rate = (
    online_cancelled
    / online_total
    * 100
)

store_rate = (
    store_cancelled
    / store_total
    * 100
)

overall_rate = (
    len(cancelled)
    / len(sales)
    * 100
)

print("\nCancellation Results")
print("-" * 70)

print("Total sales rows:", len(sales))
print("Completed rows:", len(completed))
print("Cancelled rows:", len(cancelled))

print("\nOnline")
print("Total:", online_total)
print("Cancelled:", online_cancelled)
print("Cancellation rate: {:.2f}%".format(
    online_rate
))

print("\nStore")
print("Total:", store_total)
print("Cancelled:", store_cancelled)
print("Cancellation rate: {:.2f}%".format(
    store_rate
))

print("\nOverall cancellation rate: {:.2f}%".format(
    overall_rate
))

print("\nOutput folder:")
print(OUTPUT_DIR)

# ---------------------------------------------------------
# Validation
# ---------------------------------------------------------

total_rows_valid = (
    len(sales) == 2_118_200
)

total_cancellation_valid = (
    len(cancelled) == TOTAL_CANCELLATIONS
)

online_higher = (
    online_rate > store_rate
)

statuses_valid = (
    set(sales["order_status"].unique())
    <= {"Completed", "Cancelled"}
)

if (
    total_rows_valid
    and total_cancellation_valid
    and online_higher
    and statuses_valid
):
    print("\n02.16 Step 3 validation: PASSED")
else:
    print("\n02.16 Step 3 validation: FAILED")

02.16 STEP 3 — CORRECT CANCELLATION BEHAVIOR
Sales files: 32
Total sales rows: 2118200

Cancellation Results
----------------------------------------------------------------------
Total sales rows: 2118200
Completed rows: 2033472
Cancelled rows: 84728

Online
Total: 529575
Cancelled: 31774
Cancellation rate: 6.00%

Store
Total: 1588625
Cancelled: 52954
Cancellation rate: 3.33%

Overall cancellation rate: 4.00%

Output folder:
c:\Users\HP\OneDrive\Desktop\retail_sales_analytics\data\synthetic\cancellation_adjusted\sales_adjusted

02.16 Step 3 validation: PASSED


In [8]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path.cwd().parent

INPUT_DIR = (
    BASE_DIR
    / "data"
    / "synthetic"
    / "cancellation_adjusted"
    / "sales_adjusted"
)

sales_files = sorted(INPUT_DIR.glob("sales_*.csv"))

sales_list = []

for file in sales_files:
    df = pd.read_csv(file)
    sales_list.append(df)

sales = pd.concat(
    sales_list,
    ignore_index=True
)

print("02.16 STEP 4 — CANCELLATION FINANCIAL TREATMENT VALIDATION")
print("=" * 70)

# ---------------------------------------------------------
# 1. Basic validation
# ---------------------------------------------------------

print("\nData Volume")
print("-" * 70)

print("Sales files:", len(sales_files))
print("Sales rows:", len(sales))

# ---------------------------------------------------------
# 2. Separate order statuses
# ---------------------------------------------------------

cancelled = sales[
    sales["order_status"] == "Cancelled"
].copy()

completed = sales[
    sales["order_status"] == "Completed"
].copy()

print("\nOrder Status")
print("-" * 70)

print("Completed rows:", len(completed))
print("Cancelled rows:", len(cancelled))

# ---------------------------------------------------------
# 3. Calculate financial values for cancelled records
# ---------------------------------------------------------

cancelled["gross_sales_value"] = (
    cancelled["quantity"]
    * cancelled["unit_selling_price"]
)

cancelled["discount_value"] = (
    cancelled["gross_sales_value"]
    * cancelled["discount_pct"]
    / 100
)

cancelled["net_sales_value"] = (
    cancelled["gross_sales_value"]
    - cancelled["discount_value"]
)

# ---------------------------------------------------------
# 4. Financial validation
# ---------------------------------------------------------

missing_quantity = (
    cancelled["quantity"].isna().sum()
)

missing_price = (
    cancelled["unit_selling_price"]
    .isna()
    .sum()
)

missing_discount = (
    cancelled["discount_pct"]
    .isna()
    .sum()
)

invalid_quantity = (
    cancelled["quantity"] <= 0
).sum()

invalid_price = (
    cancelled["unit_selling_price"] <= 0
).sum()

invalid_discount = (
    (cancelled["discount_pct"] < 0)
    | (cancelled["discount_pct"] > 60)
).sum()

negative_net_sales = (
    cancelled["net_sales_value"] < 0
).sum()

print("\nCancelled Transaction Financial Validation")
print("-" * 70)

print("Missing quantity:", missing_quantity)
print("Missing selling price:", missing_price)
print("Missing discount:", missing_discount)

print("Invalid quantity:", invalid_quantity)
print("Invalid selling price:", invalid_price)
print("Invalid discount:", invalid_discount)

print("Negative net sales:", negative_net_sales)

# ---------------------------------------------------------
# 5. Financial summary by channel
# ---------------------------------------------------------

channel_summary = (
    cancelled
    .groupby("sales_channel")
    .agg(
        cancelled_lines=("sales_line_id", "count"),
        cancelled_gross_sales=("gross_sales_value", "sum"),
        cancelled_discount=("discount_value", "sum"),
        cancelled_net_sales=("net_sales_value", "sum")
    )
    .reset_index()
)

print("\nCancelled Financial Summary by Channel")
print("-" * 70)

print(
    channel_summary.round(2).to_string(index=False)
)

# ---------------------------------------------------------
# 6. Validate mandatory identifiers
# ---------------------------------------------------------

mandatory_identifier_columns = [
    "sales_line_id",
    "order_id",
    "product_id",
    "store_id",
    "sales_channel"
]

missing_identifier_values = (
    cancelled[mandatory_identifier_columns]
    .isna()
    .sum()
    .sum()
)

print("\nMandatory Identifier Validation")
print("-" * 70)

print(
    "Mandatory identifier columns:",
    mandatory_identifier_columns
)

print(
    "Missing mandatory identifier values:",
    missing_identifier_values
)

# Customer ID is intentionally NOT treated as mandatory
customer_missing_count = (
    cancelled["customer_id"].isna().sum()
)

print(
    "Missing customer_id values "
    "(allowed anonymous transactions):",
    customer_missing_count
)

# ---------------------------------------------------------
# 7. Validate cancellation status
# ---------------------------------------------------------

valid_statuses = (
    set(sales["order_status"].dropna().unique())
    <= {"Completed", "Cancelled"}
)

print("\nStatus Validation")
print("-" * 70)

print(
    "Valid order statuses:",
    valid_statuses
)

# ---------------------------------------------------------
# 8. Final validation
# ---------------------------------------------------------

file_count_valid = (
    len(sales_files) == 32
)

row_count_valid = (
    len(sales) == 2_118_200
)

cancelled_exists = (
    len(cancelled) > 0
)

financial_fields_valid = (
    missing_quantity == 0
    and missing_price == 0
    and missing_discount == 0
    and invalid_quantity == 0
    and invalid_price == 0
    and invalid_discount == 0
    and negative_net_sales == 0
)

identifiers_valid = (
    missing_identifier_values == 0
)

if (
    file_count_valid
    and row_count_valid
    and cancelled_exists
    and financial_fields_valid
    and identifiers_valid
    and valid_statuses
):
    print("\n02.16 Step 4 validation: PASSED")
else:
    print("\n02.16 Step 4 validation: FAILED")

02.16 STEP 4 — CANCELLATION FINANCIAL TREATMENT VALIDATION

Data Volume
----------------------------------------------------------------------
Sales files: 32
Sales rows: 2118200

Order Status
----------------------------------------------------------------------
Completed rows: 2033472
Cancelled rows: 84728

Cancelled Transaction Financial Validation
----------------------------------------------------------------------
Missing quantity: 0
Missing selling price: 0
Missing discount: 0
Invalid quantity: 0
Invalid selling price: 0
Invalid discount: 0
Negative net sales: 0

Cancelled Financial Summary by Channel
----------------------------------------------------------------------
sales_channel  cancelled_lines  cancelled_gross_sales  cancelled_discount  cancelled_net_sales
       Online            31774           1.564928e+09         18762325.25         1.546165e+09
        Store            52954           2.607083e+09         22680013.94         2.584403e+09

Mandatory Identifier Valid

In [9]:
from pathlib import Path
import pandas as pd

# ============================================================
# 02.16 STEP 5 — FINAL CANCELLATION VALIDATION
# ============================================================

OUTPUT_DIR = Path("../data/synthetic/cancellation_adjusted/sales_adjusted")

sales_files = sorted(OUTPUT_DIR.glob("sales_*.csv"))

total_rows = 0
status_values = set()
channel_status = {}

for file in sales_files:
    df = pd.read_csv(file)

    total_rows += len(df)

    status_values.update(df["order_status"].dropna().unique())

    for channel in df["sales_channel"].dropna().unique():
        channel_df = df[df["sales_channel"] == channel]

        channel_status.setdefault(
            channel,
            {"total": 0, "cancelled": 0}
        )

        channel_status[channel]["total"] += len(channel_df)
        channel_status[channel]["cancelled"] += (
            channel_df["order_status"].eq("Cancelled").sum()
        )

# ------------------------------------------------------------
# Cancellation rates
# ------------------------------------------------------------

overall_cancelled = 0

for file in sales_files:
    df = pd.read_csv(file)
    overall_cancelled += df["order_status"].eq("Cancelled").sum()

overall_rate = (overall_cancelled / total_rows) * 100

print("02.16 STEP 5 — FINAL CANCELLATION VALIDATION")
print("=" * 65)

print("\nData Volume")
print("-" * 65)

print("Sales files:", len(sales_files))
print("Sales rows:", total_rows)

print("\nOrder Status")
print("-" * 65)

print("Status values:", sorted(status_values))
print("Cancelled rows:", overall_cancelled)
print("Completed rows:", total_rows - overall_cancelled)
print("Overall cancellation rate:", round(overall_rate, 2), "%")

print("\nCancellation Behavior by Channel")
print("-" * 65)

channel_rates_valid = True

for channel, values in channel_status.items():

    rate = (values["cancelled"] / values["total"]) * 100

    print(
        channel,
        "| Total:", values["total"],
        "| Cancelled:", values["cancelled"],
        "| Rate:", round(rate, 2), "%"
    )

    if rate <= 0 or rate >= 100:
        channel_rates_valid = False

# ------------------------------------------------------------
# Final validation
# ------------------------------------------------------------

files_valid = len(sales_files) == 32
rows_valid = total_rows == 2_118_200
statuses_valid = {"Completed", "Cancelled"}.issubset(status_values)
rate_valid = 0 < overall_rate < 10

print("\nFinal Validation")
print("-" * 65)

print("32 sales files:", files_valid)
print("2,118,200 sales rows:", rows_valid)
print("Required statuses present:", statuses_valid)
print("Overall cancellation rate valid:", rate_valid)
print("Channel cancellation rates valid:", channel_rates_valid)

if (
    files_valid
    and rows_valid
    and statuses_valid
    and rate_valid
    and channel_rates_valid
):
    print("\n02.16 Step 5 validation: PASSED")
else:
    print("\n02.16 Step 5 validation: FAILED")

02.16 STEP 5 — FINAL CANCELLATION VALIDATION

Data Volume
-----------------------------------------------------------------
Sales files: 32
Sales rows: 2118200

Order Status
-----------------------------------------------------------------
Status values: ['Cancelled', 'Completed']
Cancelled rows: 84728
Completed rows: 2033472
Overall cancellation rate: 4.0 %

Cancellation Behavior by Channel
-----------------------------------------------------------------
Store | Total: 1588625 | Cancelled: 52954 | Rate: 3.33 %
Online | Total: 529575 | Cancelled: 31774 | Rate: 6.0 %

Final Validation
-----------------------------------------------------------------
32 sales files: True
2,118,200 sales rows: True
Required statuses present: True
Overall cancellation rate valid: True
Channel cancellation rates valid: True

02.16 Step 5 validation: PASSED
